# EEG/tES Montage Similarity Scoring

This Colab notebook compares an original/ideal electrode montage against candidate replacement montages on a 2D EasyCap/Soterix cap layout.

Scope: geometry-only scoring. No electric-field modeling, no 3D coordinates, no optimization libraries, no GUI features, and no file saving.

## 1. Install/Import Dependencies

Colab usually has pandas installed. Shapely is used for circular electrode footprint overlap.

In [ ]:
try:
    import shapely
except ImportError:
    !pip install shapely

import math
import warnings
from pathlib import Path

import pandas as pd
from shapely.geometry import Point
from shapely.ops import unary_union

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

## 2. Load Coordinate CSV From Drive

Put your coordinate CSV in your designated Drive folder. If the default path is wrong, this cell searches Drive for likely coordinate CSV files and prints options you can paste into `COORDINATE_FILE`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your file has a different name or folder.
COORDINATE_FILE = '/content/drive/MyDrive/EEG_Project/Cap_Data/easycap_cac64_soterix_finetuned_final.csv'

def find_candidate_coordinate_files(search_root='/content/drive/MyDrive'):
    root = Path(search_root)
    if not root.exists():
        return []
    csv_files = list(root.rglob('*.csv'))
    likely_terms = ['easycap', 'soterix', 'coordinate', 'coords', 'cap']
    likely = [p for p in csv_files if any(term in p.name.lower() for term in likely_terms)]
    return likely or csv_files

def load_coords(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        candidates = find_candidate_coordinate_files()
        message = f'Coordinate CSV not found: {csv_path}'
        if candidates:
            message += '\n\nPossible CSV files found in Drive:'
            for i, candidate in enumerate(candidates[:20], start=1):
                message += f'\n{i}. {candidate}'
            message += '\n\nCopy the correct path above into COORDINATE_FILE, then rerun this cell.'
        else:
            message += '\n\nNo CSV files were found in /content/drive/MyDrive.'
        raise FileNotFoundError(message)

    coords = pd.read_csv(csv_path)
    required = {'label', 'x', 'y', 'status'}
    missing = required - set(coords.columns)
    if missing:
        raise ValueError(f'Missing required coordinate columns: {sorted(missing)}')

    coords = coords.copy()
    coords['label'] = coords['label'].astype(str).str.strip()
    coords['status'] = coords['status'].astype(str).str.lower().str.strip()
    return coords

coords = load_coords(COORDINATE_FILE)
print(coords['status'].value_counts())
coords.head()

## 3. Helper Functions

In [ ]:
DEFAULT_WEIGHTS = {
    'angle': 0.20,
    'distance': 0.30,
    'coverage_loss': 0.30,
    'new_area': 0.15,
    'symmetry': 0.05,
}

def get_xy(coords, label):
    matches = coords.loc[coords['label'] == label]
    if matches.empty:
        raise ValueError(f'Label not found in coordinate table: {label}')
    if len(matches) > 1:
        raise ValueError(f'Duplicate label in coordinate table: {label}')

    row = matches.iloc[0]
    return (float(row['x']), float(row['y']))

def labels_to_points(coords, labels):
    return [get_xy(coords, label) for label in labels]

def validate_equal_lengths(original_labels, candidate_labels):
    if len(original_labels) != len(candidate_labels):
        raise ValueError('Original and candidate montages must have the same number of electrodes')

def validate_two_electrode_montage(points):
    if len(points) != 2:
        raise ValueError('This metric currently supports only two-electrode montages')

def validate_candidate_status(coords, candidate_labels, strict_open=False):
    for label in candidate_labels:
        matches = coords.loc[coords['label'] == label]
        if matches.empty:
            raise ValueError(f'Label not found in coordinate table: {label}')

        status = str(matches.iloc[0]['status']).lower().strip()
        if status != 'open':
            message = f"Candidate site {label!r} has status {status!r}; candidate stimulation sites should usually be 'open'."
            if strict_open:
                raise ValueError(message)
            warnings.warn(message, UserWarning)

def angle_of_vector(p1, p2):
    return math.degrees(math.atan2(p2[1] - p1[1], p2[0] - p1[0]))

def angle_difference_deg(angle1, angle2):
    diff = abs(angle1 - angle2) % 360
    return min(diff, 360 - diff)

def montage_angle_change(original_points, candidate_points):
    validate_two_electrode_montage(original_points)
    validate_two_electrode_montage(candidate_points)
    original_angle = angle_of_vector(original_points[0], original_points[1])
    candidate_angle = angle_of_vector(candidate_points[0], candidate_points[1])
    return angle_difference_deg(original_angle, candidate_angle)

def point_distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def paired_distances_from_original(original_points, candidate_points):
    if len(original_points) != len(candidate_points):
        raise ValueError('Original and candidate point lists must have the same length')
    return [point_distance(original, candidate) for original, candidate in zip(original_points, candidate_points)]

def mean_distance_from_original(original_points, candidate_points):
    distances = paired_distances_from_original(original_points, candidate_points)
    return sum(distances) / len(distances)

def max_distance_from_original(original_points, candidate_points):
    return max(paired_distances_from_original(original_points, candidate_points))

def electrode_area(points, electrode_radius):
    circles = [Point(x, y).buffer(electrode_radius) for x, y in points]
    if not circles:
        raise ValueError('Cannot compute area for an empty montage')
    return unary_union(circles)

def area_overlap_metrics(original_points, candidate_points, electrode_radius=0.45):
    original_shape = electrode_area(original_points, electrode_radius)
    candidate_shape = electrode_area(candidate_points, electrode_radius)
    overlap_area = original_shape.intersection(candidate_shape).area
    original_area = original_shape.area
    candidate_area = candidate_shape.area
    new_area = candidate_area - overlap_area

    return {
        'original_area': original_area,
        'candidate_area': candidate_area,
        'overlap_area': overlap_area,
        'percent_original_area_covered': (overlap_area / original_area) * 100,
        'new_area': new_area,
        'percent_candidate_area_new': (new_area / candidate_area) * 100,
    }

def symmetry_error(candidate_points, symmetry_axis_x=0):
    validate_two_electrode_montage(candidate_points)
    x1 = candidate_points[0][0] - symmetry_axis_x
    x2 = candidate_points[1][0] - symmetry_axis_x
    return abs(x1 + x2)

def normalized_score(angle_change_deg, mean_distance, percent_original_area_covered, percent_candidate_area_new, symmetry, weights=None, use_symmetry=True):
    weights = DEFAULT_WEIGHTS if weights is None else weights
    angle_penalty = min(angle_change_deg / 90, 1)
    distance_penalty = min(mean_distance / 2.0, 1)
    coverage_loss_penalty = (100 - percent_original_area_covered) / 100
    new_area_penalty = percent_candidate_area_new / 100
    symmetry_penalty = min(symmetry / 2.0, 1) if use_symmetry else 0

    return (
        weights['angle'] * angle_penalty
        + weights['distance'] * distance_penalty
        + weights['coverage_loss'] * coverage_loss_penalty
        + weights['new_area'] * new_area_penalty
        + weights['symmetry'] * symmetry_penalty
    )

## 4. Scoring Functions

In [ ]:
def score_candidate_montage(
    coords,
    original_labels,
    candidate_labels,
    electrode_radius=0.45,
    use_symmetry=True,
    symmetry_axis_x=0,
    weights=None,
    strict_open=False,
):
    validate_equal_lengths(original_labels, candidate_labels)
    validate_candidate_status(coords, candidate_labels, strict_open=strict_open)

    original_points = labels_to_points(coords, original_labels)
    candidate_points = labels_to_points(coords, candidate_labels)

    angle_change = montage_angle_change(original_points, candidate_points)
    mean_distance = mean_distance_from_original(original_points, candidate_points)
    max_distance = max_distance_from_original(original_points, candidate_points)
    area_metrics = area_overlap_metrics(original_points, candidate_points, electrode_radius=electrode_radius)
    candidate_symmetry_error = symmetry_error(candidate_points, symmetry_axis_x=symmetry_axis_x) if use_symmetry else 0

    final_score = normalized_score(
        angle_change_deg=angle_change,
        mean_distance=mean_distance,
        percent_original_area_covered=area_metrics['percent_original_area_covered'],
        percent_candidate_area_new=area_metrics['percent_candidate_area_new'],
        symmetry=candidate_symmetry_error,
        weights=weights,
        use_symmetry=use_symmetry,
    )

    return {
        'original_montage': ', '.join(original_labels),
        'candidate_montage': ', '.join(candidate_labels),
        'angle_change_deg': angle_change,
        'mean_distance_from_original': mean_distance,
        'max_distance_from_original': max_distance,
        'percent_original_area_covered': area_metrics['percent_original_area_covered'],
        'percent_candidate_area_new': area_metrics['percent_candidate_area_new'],
        'new_area': area_metrics['new_area'],
        'symmetry_error': candidate_symmetry_error,
        'final_score': final_score,
    }

def score_candidate_montages(
    coords,
    original_labels,
    candidate_montages,
    electrode_radius=0.45,
    use_symmetry=True,
    symmetry_axis_x=0,
    weights=None,
    strict_open=False,
):
    rows = [
        score_candidate_montage(
            coords=coords,
            original_labels=original_labels,
            candidate_labels=candidate_labels,
            electrode_radius=electrode_radius,
            use_symmetry=use_symmetry,
            symmetry_axis_x=symmetry_axis_x,
            weights=weights,
            strict_open=strict_open,
        )
        for candidate_labels in candidate_montages
    ]

    results = pd.DataFrame(rows).sort_values('final_score').reset_index(drop=True)
    results.insert(0, 'rank', range(1, len(results) + 1))
    return results

## 5. Define Original and Candidate Montages

The original montage can use blocked EEG labels because it represents the ideal/ROAST montage. Candidate montages should usually use open Soterix labels.

In [ ]:
original_labels = ['F3', 'F4']

candidate_montages = [
    ['FFC3h', 'FFC4h'],
    ['FCC3h', 'FCC4h'],
    ['AFF5h', 'AFF6h'],
    ['CCP3h', 'CCP4h'],
]

original_labels, candidate_montages

## 6. Score Candidate Montages

In [ ]:
scores_df = score_candidate_montages(
    coords=coords,
    original_labels=original_labels,
    candidate_montages=candidate_montages,
    electrode_radius=0.45,
    use_symmetry=True,
    strict_open=False,
)

scores_df

## 7. Inspect Best Candidate

Lower `final_score` is better. The table remains in memory as `scores_df`; nothing is saved to Drive.

In [ ]:
best = scores_df.iloc[0]
print('Best candidate montage:')
print(best['candidate_montage'])
print('\nFull metric row:')
display(best.to_frame().T)